# Custom Tools Are Secretly In-Process MCP Servers

Custom tools registered with the SDK are implemented as **in-process MCP servers** running directly inside your Python application — no separate process, no subprocess, no transport, unlike the external MCP servers regular MCP setups require. `@tool` builds an MCP tool schema behind the scenes, and `create_sdk_mcp_server` wraps it in a real MCP `Server` instance.


In [1]:
from typing import Any

from claude_agent_sdk import (
    tool,  # decorator that turns a Python function into a Claude-usable tool
    create_sdk_mcp_server,  # bundles one or more tools into a "server" Claude can talk to
)


def get_stock_price(ticker: str) -> dict[str, float]:
    """Mock stock price lookup — no real API call."""
    mock_prices = {"AAPL": 193.50, "GOOGL": 178.25, "MSFT": 412.80}
    return {"price": mock_prices.get(ticker.upper(), 100.00)}


# Same @tool setup as the last episode. This time we'll peek inside the
# objects it creates to see the MCP machinery hiding underneath.
@tool("get_stock_price", "Get the current mock stock price for a ticker symbol", {"ticker": str})
async def get_stock_price_tool(args: dict[str, Any]) -> dict[str, Any]:
    price = get_stock_price(args["ticker"])
    return {"content": [{"type": "text", "text": f"{args['ticker']}: ${price['price']:.2f}"}]}


stock_server = create_sdk_mcp_server(name="stocks", version="1.0.0", tools=[get_stock_price_tool])

## The MCP-style tool schema `@tool` generates


In [2]:
# @tool didn't just wrap our function — it attached MCP-style metadata to it,
# the same shape Claude uses to understand any tool (built-in or custom).
print("Tool name:       ", get_stock_price_tool.name)  # -> "get_stock_price"
print("Tool description:", get_stock_price_tool.description)  # -> what we wrote in @tool(...)
print("Input schema:    ", get_stock_price_tool.input_schema)  # -> {"ticker": str}, from @tool(...)

Tool name:        get_stock_price
Tool description: Get the current mock stock price for a ticker symbol
Input schema:     {'ticker': <class 'str'>}


## The server `create_sdk_mcp_server` builds — a real MCP `Server` object


In [3]:
# stock_server (from create_sdk_mcp_server) is a small dict wrapping a real
# MCP server object underneath — this is the "no separate process" part.
print("Config type:     ", stock_server["type"])  # "sdk" — an in-process MCP server, not a subprocess
print("Server name:     ", stock_server["name"])
print("Underlying object:", type(stock_server["instance"]))

Config type:      sdk
Server name:      stocks
Underlying object: <class 'mcp.server.lowlevel.server.Server'>


## Next episode

This episode's server ran entirely in-process — same Python process, no subprocess involved. Next episode: connecting to an **external** MCP server, one that runs as its own separate process and gets configured rather than written in Python.
